# Beam search

- technique which involves **limiting** the queue in which a search algorithm stores it values
  
  

## UCS Beam search

In [ ]:
import heapq
def beam_search(start, goal, beam_width=2):
    # Initialize the beam with the start state
    beam = [(0, [start])]  # (cumulative cost, path)

    while beam:
        candidates = []
        # Expand each path in the beam
        for cost, path in beam:
            current_node = path[-1]
            if current_node == goal:
                return path, cost  # Return the path and cost if goal
            # Generate successors
            for neighbor, edge_cost in graph.get(current_node, []):
                new_cost = cost + edge_cost
                new_path = path + [neighbor]
                candidates.append((new_cost, new_path))
        # Select top-k paths based on the lowest cumulative cost
        beam = heapq.nsmallest(beam_width, candidates, key=lambda x: x[0])
        #print(beam)
    return None, float('inf')  # Return None if no path is found 

### 1. The Core Data Structures (Data Types)

Before looking at the logic, here is exactly what data types are flowing through this function:

* **`beam`**: A **List of Tuples** `List[Tuple[int, List[str]]]`.
* Example: `[(5, ['S', 'B']), (4, ['S', 'C'])]`
* This holds the "surviving" paths for the current depth layer.


* **`candidates`**: Also a **List of Tuples**.
* It temporarily holds *all* possible next steps before they get pruned.


* **`graph`** *(Implicit Global Variable)*: A **Dictionary** `Dict[str, List[Tuple[str, int]]]`.
* Example: `{'S': [('A', 2), ('B', 5)]}`. The function assumes this exists outside the function.



---

### 2. Step-by-Step Logic Breakdown

Here is exactly what each line is doing and how the methods work.

#### Step 1: Initialization

```python
beam = [(0, [start])]

```

* You initialize the `beam` as a list containing one tuple.
* `0` is the starting integer cost. `[start]` is a list of strings containing just the first node (e.g., `['S']`).

#### Step 2: The Main Loop & Candidate Pool

```python
while beam:
    candidates = []

```

* `while beam:` means "keep looping as long as the beam list is not empty." If the beam goes empty, it means we hit dead ends and there is no path to the goal.
* `candidates = []` creates a fresh, empty list for the next layer. We do this at the start of every loop so we don't mix Layer 1 nodes with Layer 2 nodes.

#### Step 3: Expanding the Beam

```python
    for cost, path in beam:
        current_node = path[-1]

```

* **Unpacking:** `for cost, path` takes the tuple `(0, ['S'])` and splits it so `cost = 0` and `path = ['S']`.
* **List Indexing (`[-1]`):** This is a Python trick to get the very last item in a list. If `path` is `['S', 'A', 'D']`, `path[-1]` returns `'D'`. This is the node we are currently standing on.

#### Step 4: The Goal Check

```python
        if current_node == goal:
            return path, cost

```

* If we stepped onto the goal node, the function immediately stops and hands back the winning `path` list and the total `cost` integer. *(Note: Because this check happens here, it stops as soon as it sees the goal, acting a bit like a Greedy search at the very end!)*

#### Step 5: Generating Successors

```python
        for neighbor, edge_cost in graph.get(current_node, []):
            new_cost = cost + edge_cost
            new_path = path + [neighbor]
            candidates.append((new_cost, new_path))

```

* **`graph.get(current_node, [])`**: This searches the dictionary for the `current_node`. If the node exists, it returns its list of neighbors. If the node is a dead end and doesn't exist in the dictionary, it returns an empty list `[]` instead of crashing the program.
* **List Concatenation (`+`)**: `new_path = path + [neighbor]` is a brilliant way to handle paths. It takes the old list (e.g., `['S', 'A']`), creates a *brand new copy* in memory, and sticks the new neighbor on the end (`['S', 'A', 'D']`). This prevents different branches from accidentally editing the same path list.
* **`.append()`**: Bundles the new cost and new path into a tuple `(...)` and drops it into our temporary `candidates` pool.

#### Step 6: The Pruning (The "Beam" mechanism)

```python
    beam = heapq.nsmallest(beam_width, candidates, key=lambda x: x[0])

```

This single line is what transforms this code from a Breadth-First Search into a Beam Search!

* **`heapq.nsmallest()`**: This is a highly optimized method from Python's standard library. Instead of sorting the *entire* giant candidate list (which is slow), it efficiently scans the list and pulls out only the `n` smallest items.
* **`beam_width`**: Tells the method how many items to keep (e.g., keep the top 2).
* **`candidates`**: The pool it is searching through.
* **`key=lambda x: x[0]`**: This tells `nsmallest` *how* to judge what is "small".
* `x` represents one tuple in the candidate list, like `(15, ['S', 'B', 'T'])`.
* `x[0]` tells it to look strictly at index `0` of the tuple (the integer cost, `15`).


* It grabs the best tuples, and replaces the old `beam` with this new, pruned list.

#### Step 7: The Failure State

```python
return None, float('inf')

```

* If the `while beam:` loop finishes and we never found the goal, it returns `None` for the path and `float('inf')` (infinity) for the cost, signifying a failed search.

---

### Summary of what makes this code great

Your use of `heapq.nsmallest` is a massive upgrade over the `list.sort()` method we discussed earlier. It is much faster and more memory-efficient when dealing with hundreds of candidates.

Would you like me to show you how to write a quick test case (with a mock graph dictionary) to execute this exact function and print the output?

## Greedy BFS beam search


In [12]:
graph = {
    'S': {'A': 3, 'B': 6, 'C': 5},
    'A': {'D': 9, 'E': 8},
    'B': {'F': 12, 'G': 14},
    'C': {'H': 7},
    'H': {'I': 5, 'J': 6},
    'I': {'K': 1, 'L': 10, 'M': 2},
    'D': {}, 'E': {}, 'F': {}, 'G': {},
    'J': {}, 'K': {}, 'L': {}, 'M': {}
}

heuristic = {
    'S': 10, 'A': 9, 'B': 7, 'C': 5, 'D': 8, 'E': 6, 'F': 4, 'G': 3,
    'H': 3, 'I': 2, 'J': 6, 'K': 2, 'L': 0, 'M': 1
}


In [13]:
def beam_search(graph, start, goal, beam_width=2):
    frontier = [(start, heuristic[start])]
    visited = set()
    came_from = {start: None}
    print("came_from:",came_from)


    while frontier:
        frontier.sort(key=lambda x: x[1]) # sort asc on basis of heuristics 
        print("frontier: ",frontier)
        frontier = frontier[:beam_width]  # Keep only top beam_width nodes
        print("frontier2: ",frontier)
        next_frontier = []

        # MARK ALL CHILDREN OF ALL FRONTIER NODES AS visited
        for current_node, _ in frontier:
            if current_node in visited:
                continue
            visited.add(current_node)
            # MARK ALL CHILDREN OF ALL FRONTIER NODES AS visited
            
            # if goal is found reconstruct the path to source
            if current_node == goal:
                path = []
                while current_node is not None:
                    path.append(current_node)
                    print("path:",path)
                    current_node = came_from[current_node]
                path.reverse()
                print(f"Goal found with Beam Search. Path: {path}")
                return
            # if goal is found reconstruct the path to source


            
            
            for neighbor in graph[current_node]:
                if neighbor not in visited:
                    next_frontier.append((neighbor, heuristic[neighbor]))
                    came_from[neighbor] = current_node

        frontier = next_frontier

    print("Goal not found")

In [ ]:
beam_search(graph, 'S', 'L')

came_from: {'S': None}
frontier:  [('S', 10)]
frontier2:  [('S', 10)]
frontier:  [('C', 5), ('B', 7), ('A', 9)]
frontier2:  [('C', 5), ('B', 7)]
frontier:  [('H', 3), ('G', 3), ('F', 4)]
frontier2:  [('H', 3), ('G', 3)]
frontier:  [('I', 2), ('J', 6)]
frontier2:  [('I', 2), ('J', 6)]
frontier:  [('L', 0), ('M', 1), ('K', 2)]
frontier2:  [('L', 0), ('M', 1)]
path: ['L']
path: ['L', 'I']
path: ['L', 'I', 'H']
path: ['L', 'I', 'H', 'C']
path: ['L', 'I', 'H', 'C', 'S']
Goal found with Beam Search. Path: ['S', 'C', 'H', 'I', 'L']


### 1. The Core Data Structures (Data Types)

Before looking at the logic, here is exactly what data types are flowing through this function.

* **`frontier`**: A **List of Tuples** `List[Tuple[str, int]]`.
* Example: `[('S', 10), ('A', 9)]`
* This holds the "surviving" nodes for the current depth layer, paired with their heuristic score (the estimated distance to the goal).


* **`next_frontier`**: Also a **List of Tuples**.
* It temporarily holds all the newly discovered neighbors before they replace the main `frontier`.


* **`visited`**: A **Set of Strings** `Set[str]`.
* Example: `{'S', 'B'}`
* Sets in Python are highly optimized for checking if an item exists inside them. This prevents the algorithm from getting stuck in an infinite loop if nodes connect back to each other.


* **`came_from`**: A **Dictionary** `Dict[str, str]`.
* Example: `{'A': 'S', 'D': 'A', 'S': None}`
* This acts as a breadcrumb trail. It links a child node (the key) to the parent node that discovered it (the value).


* **`graph`** *(Implicit Global Variable)*: A **Dictionary** mapping a string to a collection of strings `Dict[str, List[str]]` or `Dict[str, Dict]`.
* Example: `{'S': ['A', 'B']}`. The function assumes this maps nodes to their neighbors.


* **`heuristic`** *(Implicit Global Variable)*: A **Dictionary** mapping strings to integers `Dict[str, int]`.
* Example: `{'S': 10, 'Goal': 0}`. The function uses this as a lookup table for the node scores.



---

### 2. Step-by-Step Logic Breakdown

Here is exactly what each line is doing and how the Python methods work.

#### Step 1: Initialization

```python
    frontier = [(start, heuristic[start])]
    visited = set()
    came_from = {start: None}

```

* `[(start, heuristic[start])]`: Creates the initial list with one tuple. It looks up the start node's score in the `heuristic` dictionary (e.g., `('S', 10)`).
* `set()`: Initializes an empty mathematical set for tracking memory.
* `{start: None}`: Initializes the dictionary. It records that the start node has no parent (`None`) because it is the beginning of the path.

#### Step 2: Sorting the Frontier

```python
    while frontier:
        frontier.sort(key=lambda x: x[1])

```

* `while frontier:`: The loop continues running as long as the `frontier` list is not empty. If it empties out, the search has hit a dead end.
* **`.sort()`**: This method reorganizes the list in place.
* **`key=lambda x: x[1]`**: This tells Python *how* to sort the tuples.
* `x` represents a single tuple, like `('A', 9)`.
* `x[1]` tells it to look strictly at index `1` (the integer score `9`).
* This sorts the list from the lowest heuristic score to the highest.



#### Step 3: Pruning (The "Beam" mechanism)

```python
        frontier = frontier[:beam_width]  # Keep only top beam_width nodes
        next_frontier = []

```

* **List Slicing (`[:beam_width]`)**: This is Python syntax for taking a slice of a list. If `beam_width` is 2, `[:2]` creates a new list containing only the items at index `0` and index `1`. The rest of the nodes are permanently discarded.
* `next_frontier = []`: Creates an empty list to stage the next layer of nodes.

#### Step 4: Unpacking and Checking Memory

```python
        for current_node, _ in frontier:
            if current_node in visited:
                continue
            visited.add(current_node)

```

* **Tuple Unpacking (`current_node, _`)**: The `for` loop grabs a tuple like `('A', 9)`. It assigns `'A'` to `current_node`. The underscore `_` is a Python convention that means "I have to unpack this value (the 9), but I am going to throw it away and not use it."
* **`in visited`**: Checks if the string name is already inside the set.
* **`continue`**: If it is in the set, `continue` forces the loop to instantly skip the rest of the code block and move to the next item in the `frontier`.
* **`.add()`**: If it wasn't skipped, it gets added to the `visited` set so it won't be processed again in the future.

#### Step 5: The Goal Check & Path Reconstruction

```python
            if current_node == goal:
                path = []
                while current_node is not None:
                    path.append(current_node)
                    current_node = came_from[current_node]
                path.reverse()
                print(f"Goal found with Beam Search. Path: {path}")
                return

```

* `if current_node == goal:`: Checks if the current string matches the target.
* `while current_node is not None:`: This loop traces the breadcrumbs backward. It starts at the goal node and stops when it hits the start node (which we initialized with a parent of `None`).
* **`.append()`**: Adds the node to the `path` list.
* **`came_from[current_node]`**: Looks up the current node in the dictionary to find its parent, and then updates the `current_node` variable to that parent so the loop can step backward.
* **`.reverse()`**: Because we traced from the Goal backward to the Start, the `path` list is backwards. `.reverse()` flips it in place so it reads from Start to Goal.
* `return`: Immediately stops the entire function.

#### Step 6: Generating Successors

```python
            for neighbor in graph[current_node]:
                if neighbor not in visited:
                    next_frontier.append((neighbor, heuristic[neighbor]))
                    came_from[neighbor] = current_node

```

* `graph[current_node]`: Looks up the current node in the graph dictionary to get its neighbors.
* `if neighbor not in visited:`: Ensures we don't queue up a node we've already explored.
* **`.append((...))`**: Looks up the neighbor's score in the `heuristic` dictionary, bundles the name and score into a tuple, and drops it into the `next_frontier` list.
* `came_from[neighbor] = current_node`: Writes a new entry in the dictionary, linking the new neighbor back to the node we are currently standing on.

#### Step 7: Advancing the Search and Failure State

```python
        frontier = next_frontier

    print("Goal not found")

```

* `frontier = next_frontier`: Once the `for` loop finishes evaluating the current layer, it takes the newly generated `next_frontier` and completely overwrites the old `frontier`. The `while` loop then restarts from the top to sort and slice this new layer.
* `print("Goal not found")`: If the `while` loop eventually runs out of nodes (because the `frontier` becomes empty), the function drops down to this final line and prints the failure message.

## simpler GBFS

In [15]:
def beam_search(graph, start, goal, beam_width=2):
    # 1. Simplified Frontier: Only store node names, not tuples
    frontier = [start]  
    visited = set()
    came_from = {start: None}

    while frontier:
        # 2. Simplified Sorting: Look up the score directly in the lambda function
        frontier.sort(key=lambda node: heuristic[node])

        frontier = frontier[:beam_width]  
        print("frontier2:",frontier)
        next_frontier = []

        # 3. Simplified Unpacking: No need for 'current_node, _'
        for current_node in frontier:
            if current_node in visited:
                continue
            visited.add(current_node)

            if current_node == goal:
                path = []
                while current_node is not None:
                    path.append(current_node)
                    print("path:",path)
                    current_node = came_from[current_node]
                path.reverse()
                print(f"Goal found with Beam Search. Path: {path}")
                return

            # 4. Safe Lookup: .get() prevents crashes if a node has no neighbors
            for neighbor in graph.get(current_node, []):
                if neighbor not in visited:
                    # 5. Simplified Generation: Just append the string
                    next_frontier.append(neighbor)  
                    came_from[neighbor] = current_node

        frontier = next_frontier

    print("Goal not found")

In [16]:
beam_search(graph, 'S', 'L')

frontier2: ['S']
frontier2: ['C', 'B']
frontier2: ['H', 'G']
frontier2: ['I', 'J']
frontier2: ['L', 'M']
path: ['L']
path: ['L', 'I']
path: ['L', 'I', 'H']
path: ['L', 'I', 'H', 'C']
path: ['L', 'I', 'H', 'C', 'S']
Goal found with Beam Search. Path: ['S', 'C', 'H', 'I', 'L']
